# Distância de Manhattan — Ground Truth × Gerações Gherkin (JSON)

Notebook adaptado para o novo formato de dados:

- **Ground truth**: JSON com `cases`, em que cada caso possui `case_id`, `original_case`, `reference_id` e `gherkin`.
- **Gerações**: JSON com metadados `model`, `technique`, `number_of_executions` e, para cada caso, uma lista `generations`.
- A associação entre referência e geração é feita por **`case_id`**, não pela posição do caso no arquivo.
- O cálculo de **Distância de Manhattan** mantém a lógica do notebook original: cada par de textos é vetorizado com `CountVectorizer` e comparado com `cityblock`.
- O notebook aceita **1 ground truth e 1 ou mais arquivos de gerações** no mesmo upload.
- Ao final, é gerado um **CSV detalhado** com todas as comparações e rankings.

> **Interpretação:** quanto menor a Distância de Manhattan, mais próximos são os vetores de contagem de termos dos dois cenários, de acordo com esta representação.

In [ ]:
# ============================================================
# 1. IMPORTAÇÕES E CONFIGURAÇÃO
# ============================================================

import json
import re
import warnings

import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer
from scipy.spatial.distance import cityblock

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", None)

# Se True, o CSV será baixado automaticamente ao final no Google Colab.
BAIXAR_CSV_AUTOMATICAMENTE = True

In [ ]:
# ============================================================
# 2. UPLOAD E IDENTIFICAÇÃO AUTOMÁTICA DOS JSONs
# ============================================================

def carregar_json_bytes(nome_arquivo, conteudo):
    # Aceita UTF-8 com ou sem BOM.
    try:
        texto = conteudo.decode("utf-8-sig")
        return json.loads(texto)
    except Exception as e:
        raise ValueError(f"Não foi possível ler '{nome_arquivo}' como JSON: {e}") from e


def classificar_json(nome_arquivo, dados):
    # Classifica como ground_truth, geracoes ou desconhecido.
    if not isinstance(dados, dict):
        return "desconhecido"

    casos = dados.get("cases")
    if not isinstance(casos, list):
        return "desconhecido"

    if dados.get("is_reference_base") is True:
        return "ground_truth"

    if casos:
        primeiro = casos[0]
        if isinstance(primeiro, dict):
            if "generations" in primeiro:
                return "geracoes"
            if "reference_id" in primeiro and "gherkin" in primeiro:
                return "ground_truth"

    return "desconhecido"


def processar_upload(uploaded):
    arquivos = []
    for nome, conteudo in uploaded.items():
        if not nome.lower().endswith(".json"):
            print(f"⚠ Ignorado (não é JSON): {nome}")
            continue

        dados = carregar_json_bytes(nome, conteudo)
        tipo = classificar_json(nome, dados)
        arquivos.append({"nome": nome, "tipo": tipo, "dados": dados})
    return arquivos


try:
    from google.colab import files
except ImportError as e:
    raise RuntimeError(
        "Este notebook foi preparado para upload interativo no Google Colab. "
        "Execute-o no Colab ou adapte esta célula para leitura local."
    ) from e

# ------------------------------------------------------------
# ETAPA 1 — Ground truth
# ------------------------------------------------------------
print("ETAPA 1/2 — Envie o arquivo JSON do ground truth:")
upload_gt = files.upload()
arquivos_json = processar_upload(upload_gt)

ground_truths = [a for a in arquivos_json if a["tipo"] == "ground_truth"]
arquivos_geracoes = [a for a in arquivos_json if a["tipo"] == "geracoes"]
desconhecidos = [a["nome"] for a in arquivos_json if a["tipo"] == "desconhecido"]

if desconhecidos:
    print("⚠ JSON(s) com estrutura não reconhecida:", desconhecidos)

if len(ground_truths) != 1:
    raise ValueError(
        f"É necessário exatamente 1 ground truth. Foram identificados {len(ground_truths)}. "
        "Verifique se o arquivo possui 'is_reference_base': true ou casos com "
        "'reference_id' e 'gherkin'."
    )

ground_truth_nome = ground_truths[0]["nome"]
ground_truth = ground_truths[0]["dados"]
print(f"\n✓ Ground truth identificado: {ground_truth_nome}")

# ------------------------------------------------------------
# ETAPA 2 — Gerações
# ------------------------------------------------------------
# Se o usuário já tiver enviado gerações junto com o ground truth, elas são
# aproveitadas. Caso contrário, abre um segundo seletor de arquivos.
if not arquivos_geracoes:
    print("\nETAPA 2/2 — Agora envie um ou mais JSONs de gerações:")
    upload_gen = files.upload()
    novos_arquivos = processar_upload(upload_gen)

    novos_ground_truths = [a for a in novos_arquivos if a["tipo"] == "ground_truth"]
    if novos_ground_truths:
        print(
            "⚠ Ground truth adicional ignorado na etapa de gerações:",
            [a["nome"] for a in novos_ground_truths]
        )

    novos_desconhecidos = [
        a["nome"] for a in novos_arquivos if a["tipo"] == "desconhecido"
    ]
    if novos_desconhecidos:
        print("⚠ JSON(s) com estrutura não reconhecida:", novos_desconhecidos)

    arquivos_geracoes.extend(
        a for a in novos_arquivos if a["tipo"] == "geracoes"
    )

if not arquivos_geracoes:
    raise ValueError(
        "Nenhum arquivo de gerações foi identificado. Os arquivos de gerações "
        "devem possuir 'cases' e, dentro de cada caso, a chave 'generations'."
    )

print(f"\n✓ Arquivos de gerações identificados: {len(arquivos_geracoes)}")
for arq in arquivos_geracoes:
    dados = arq["dados"]
    print(
        f"  - {arq['nome']} | modelo={dados.get('model')} | "
        f"técnica={dados.get('technique')} | "
        f"execuções declaradas={dados.get('number_of_executions')}"
    )


ETAPA 1/2 — Envie o arquivo JSON do ground truth:


Saving base_referencia_gherkin.json to base_referencia_gherkin.json

✓ Ground truth identificado: base_referencia_gherkin.json

ETAPA 2/2 — Agora envie um ou mais JSONs de gerações:


Saving geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json to geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json

✓ Arquivos de gerações identificados: 1
  - geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json | modelo=ibm-granite/granite-4.1-8b | técnica=few-shot | execuções declaradas=10


In [ ]:
# ============================================================
# 3. VALIDAÇÃO DOS DADOS
# ============================================================

def indexar_ground_truth(dados_ground_truth):
    refs = {}
    duplicados = []

    for caso in dados_ground_truth.get("cases", []):
        case_id = caso.get("case_id")
        if not case_id:
            continue

        if case_id in refs:
            duplicados.append(case_id)

        refs[case_id] = caso

    if duplicados:
        raise ValueError(
            f"case_id duplicado(s) no ground truth: {sorted(set(duplicados))}"
        )

    return refs


referencias = indexar_ground_truth(ground_truth)
avisos_validacao = []

for arq in arquivos_geracoes:
    nome = arq["nome"]
    dados = arq["dados"]
    ids_arquivo = []
    declaradas = dados.get("number_of_executions")

    for caso in dados.get("cases", []):
        case_id = caso.get("case_id")
        ids_arquivo.append(case_id)

        if case_id not in referencias:
            avisos_validacao.append(
                f"{nome}: {case_id} existe nas gerações, mas não no ground truth."
            )
            continue

        original_ref = referencias[case_id].get("original_case")
        original_gen = caso.get("original_case")
        if (
            original_ref is not None
            and original_gen is not None
            and original_ref != original_gen
        ):
            avisos_validacao.append(
                f"{nome}: original_case divergente em {case_id}."
            )

        generations = caso.get("generations", [])
        if declaradas is not None and len(generations) != declaradas:
            avisos_validacao.append(
                f"{nome}: {case_id} possui {len(generations)} gerações, "
                f"mas o arquivo declara {declaradas}."
            )

        execucoes = [g.get("execution") for g in generations]
        execucoes_validas = [e for e in execucoes if e is not None]

        if len(execucoes_validas) != len(set(execucoes_validas)):
            avisos_validacao.append(
                f"{nome}: há números de execução duplicados em {case_id}."
            )

    ids_ref = set(referencias)
    ids_gen = set(ids_arquivo)

    ausentes = sorted(ids_ref - ids_gen)
    if ausentes:
        avisos_validacao.append(
            f"{nome}: {len(ausentes)} case_id(s) do ground truth não aparecem nas gerações. "
            f"Exemplos: {ausentes[:10]}"
        )

print(f"Casos no ground truth: {len(referencias)}")

if avisos_validacao:
    print(f"\n⚠ Foram encontrados {len(avisos_validacao)} aviso(s) de validação:")
    for aviso in avisos_validacao:
        print(" -", aviso)
else:
    print("\n✓ Estrutura validada sem avisos.")

Casos no ground truth: 259

✓ Estrutura validada sem avisos.


In [ ]:
# ============================================================
# 4. MÉTRICA: DISTÂNCIA DE MANHATTAN
# ============================================================

def manhattan_distance(texto_referencia, texto_gerado):
    # Mantém a mesma estratégia do notebook original:
    # CountVectorizer no par de textos + cityblock (L1).
    texto_referencia = "" if texto_referencia is None else str(texto_referencia)
    texto_gerado = "" if texto_gerado is None else str(texto_gerado)

    if not texto_referencia.strip() and not texto_gerado.strip():
        return 0.0

    try:
        matriz = CountVectorizer().fit_transform(
            [texto_referencia, texto_gerado]
        ).toarray()
    except ValueError as e:
        warnings.warn(
            f"Não foi possível vetorizar um par; distância definida como NaN. Erro: {e}"
        )
        return float("nan")

    return float(cityblock(matriz[0], matriz[1]))

In [ ]:
# ============================================================
# 5. CÁLCULO DAS COMPARAÇÕES
# ============================================================

resultados = []

for arq in arquivos_geracoes:
    nome_geracoes = arq["nome"]
    dados = arq["dados"]

    modelo = dados.get("model", "")
    tecnica = dados.get("technique", "")
    execucoes_declaradas = dados.get("number_of_executions")

    for caso_gerado in dados.get("cases", []):
        case_id = caso_gerado.get("case_id")
        referencia = referencias.get(case_id)

        if referencia is None:
            continue

        gherkin_ref = referencia.get("gherkin", "")

        for geracao in caso_gerado.get("generations", []):
            gherkin_gerado = geracao.get("gherkin", "")
            distancia = manhattan_distance(gherkin_ref, gherkin_gerado)

            resultados.append({
                "arquivo_ground_truth": ground_truth_nome,
                "arquivo_geracoes": nome_geracoes,
                "modelo": modelo,
                "tecnica": tecnica,
                "execucoes_declaradas": execucoes_declaradas,
                "case_id": case_id,
                "source_id": referencia.get("source_id"),
                "source_line": referencia.get("source_line"),
                "original_case": referencia.get("original_case"),
                "reference_id": referencia.get("reference_id"),
                "generation_id": geracao.get("generation_id"),
                "execucao": geracao.get("execution"),
                "distancia_manhattan": distancia,
                "gherkin_ground_truth": gherkin_ref,
                "gherkin_gerado": gherkin_gerado,
            })

df_resultados = pd.DataFrame(resultados)

if df_resultados.empty:
    raise ValueError("Nenhuma comparação pôde ser calculada.")

chaves_ranking = ["arquivo_geracoes", "modelo", "tecnica", "case_id"]

df_resultados = df_resultados.sort_values(
    chaves_ranking + ["distancia_manhattan", "execucao"],
    kind="stable",
    na_position="last"
).reset_index(drop=True)

# Assim como no notebook original, o ranking é sequencial após ordenar
# pela menor distância; empates continuam ocupando posições sucessivas.
df_resultados["ranking_no_caso"] = (
    df_resultados.groupby(chaves_ranking, dropna=False).cumcount() + 1
)

colunas = [
    "modelo",
    "tecnica",
    "case_id",
    "source_id",
    "original_case",
    "execucao",
    "distancia_manhattan",
    "ranking_no_caso",
    "generation_id",
    "reference_id",
    "gherkin_ground_truth",
    "gherkin_gerado",
    "execucoes_declaradas",
    "arquivo_ground_truth",
    "arquivo_geracoes",
]

df_resultados = df_resultados[colunas]

print(f"✓ Comparações calculadas: {len(df_resultados):,}")
print(f"✓ Casos avaliados: {df_resultados['case_id'].nunique():,}")

✓ Comparações calculadas: 2,590
✓ Casos avaliados: 259


In [ ]:
# ============================================================
# 6. TABELAS ORGANIZADAS
# ============================================================

df_resumo_geral = (
    df_resultados
    .groupby(["arquivo_geracoes", "modelo", "tecnica"], dropna=False)
    .agg(
        casos=("case_id", "nunique"),
        comparacoes=("distancia_manhattan", "count"),
        distancia_media=("distancia_manhattan", "mean"),
        distancia_mediana=("distancia_manhattan", "median"),
        desvio_padrao=("distancia_manhattan", "std"),
        distancia_minima=("distancia_manhattan", "min"),
        distancia_maxima=("distancia_manhattan", "max"),
    )
    .reset_index()
)

print("RESUMO GERAL")
display(
    df_resumo_geral.style.format({
        "distancia_media": "{:.3f}",
        "distancia_mediana": "{:.3f}",
        "desvio_padrao": "{:.3f}",
        "distancia_minima": "{:.3f}",
        "distancia_maxima": "{:.3f}",
    })
)

df_resumo_casos = (
    df_resultados
    .groupby(
        ["arquivo_geracoes", "modelo", "tecnica", "case_id", "original_case"],
        dropna=False
    )
    .agg(
        execucoes_avaliadas=("execucao", "count"),
        distancia_media=("distancia_manhattan", "mean"),
        distancia_mediana=("distancia_manhattan", "median"),
        desvio_padrao=("distancia_manhattan", "std"),
        melhor_distancia=("distancia_manhattan", "min"),
        pior_distancia=("distancia_manhattan", "max"),
    )
    .reset_index()
    .sort_values(["modelo", "tecnica", "case_id"])
)

print("\nRESUMO POR CASO — primeiras 30 linhas")
display(
    df_resumo_casos.head(30).style.format({
        "distancia_media": "{:.3f}",
        "distancia_mediana": "{:.3f}",
        "desvio_padrao": "{:.3f}",
        "melhor_distancia": "{:.3f}",
        "pior_distancia": "{:.3f}",
    })
)

print("\nCOMPARAÇÕES DETALHADAS — primeiras 50 linhas")
colunas_visualizacao = [
    "modelo",
    "tecnica",
    "case_id",
    "original_case",
    "execucao",
    "distancia_manhattan",
    "ranking_no_caso",
]
display(
    df_resultados[colunas_visualizacao]
    .head(50)
    .style
    .format({"distancia_manhattan": "{:.3f}"})
)

RESUMO GERAL


,arquivo_geracoes,modelo,tecnica,casos,comparacoes,distancia_media,distancia_mediana,desvio_padrao,distancia_minima,distancia_maxima
0,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,259,2590,39.975,40.000,7.156,16.000,86.000



RESUMO POR CASO — primeiras 30 linhas


,arquivo_geracoes,modelo,tecnica,case_id,original_case,execucoes_avaliadas,distancia_media,distancia_mediana,desvio_padrao,melhor_distancia,pior_distancia
0,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,10,38.500,39.000,2.121,35.000,43.000
1,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1002,Cadastrar transação com campos inválidos,10,33.500,32.500,2.799,31.000,40.000
2,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1006,Verificar se todos os dados da transação estão sendo exibidos,10,48.100,47.500,2.331,45.000,52.000
3,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1007,Verificar inserção de link da transação inexistente,10,43.400,43.500,5.461,34.000,53.000
4,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_101,Validar resultado de consulta de Matéria-Prima vazia,10,34.600,34.000,2.675,32.000,41.000
5,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1030,Cadastar categoria com sucesso,10,29.200,28.000,2.974,27.000,36.000
6,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1033,Cadastrar categoria com campos inválidos,10,32.900,32.000,3.247,29.000,39.000
7,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1035,Editar categoria deixando os campos obrigatórios do formulário em branco,10,40.800,39.000,6.957,32.000,51.000
8,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1060,Cancelar cadastro de conta com sucesso,10,43.700,42.000,4.523,38.000,49.000
9,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1061,Cadastrar conta com campos obrigatórios não preenchidos,10,34.300,35.000,2.263,30.000,37.000



COMPARAÇÕES DETALHADAS — primeiras 50 linhas


,modelo,tecnica,case_id,original_case,execucao,distancia_manhattan,ranking_no_caso
0,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,4,35.000,1
1,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,10,36.000,2
2,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,6,38.000,3
3,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,9,38.000,4
4,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,1,39.000,5
5,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,2,39.000,6
6,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,3,39.000,7
7,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,7,39.000,8
8,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,8,39.000,9
9,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,5,43.000,10


In [ ]:
# ============================================================
# 7. CONSULTA RÁPIDA DE UM CASO
# ============================================================

def visualizar_caso(case_id):
    # Exibe todas as execuções de um case_id, da menor para a maior distância.
    recorte = df_resultados[df_resultados["case_id"] == case_id].copy()

    if recorte.empty:
        print(f"Nenhum resultado encontrado para {case_id}.")
        return

    colunas = [
        "modelo",
        "tecnica",
        "case_id",
        "original_case",
        "execucao",
        "distancia_manhattan",
        "ranking_no_caso",
        "gherkin_ground_truth",
        "gherkin_gerado",
    ]

    display(
        recorte[colunas]
        .sort_values(["modelo", "tecnica", "distancia_manhattan", "execucao"])
        .style
        .format({"distancia_manhattan": "{:.3f}"})
    )


primeiro_case_id = df_resultados["case_id"].iloc[0]
print(f"Exemplo de consulta: {primeiro_case_id}")
visualizar_caso(primeiro_case_id)

# Para consultar outro:
# visualizar_caso("TC_261")

Exemplo de consulta: TC_1


,modelo,tecnica,case_id,original_case,execucao,distancia_manhattan,ranking_no_caso,gherkin_ground_truth,gherkin_gerado
0,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,4,35.000,1,Scenario: Cadastrar Entidade com sucesso Given que o usuário está na funcionalidade de cadastro de entidade When ele preenche os dados e confirma o cadastro Then a entidade é cadastrada com sucesso,"Scenario: Cadastrar Entidade com sucesso Given o usuário está na página de cadastro de entidade And o usuário preenche todos os campos obrigatórios com dados válidos When o usuário clica no botão ""Salvar"" Then o sistema deve exibir uma mensagem de confirmação ""Entidade cadastrada com sucesso"" And o novo registro deve aparecer na lista de entidades existentes"
1,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,10,36.000,2,Scenario: Cadastrar Entidade com sucesso Given que o usuário está na funcionalidade de cadastro de entidade When ele preenche os dados e confirma o cadastro Then a entidade é cadastrada com sucesso,Scenario: Cadastrar Entidade com sucesso Given o usuário está na página de cadastro da entidade And o usuário preenche todos os campos obrigatórios com dados válidos When o usuário clica no botão de salvar Then o sistema deve validar os dados e exibir uma mensagem de sucesso And a entidade deve ser armazenada no banco de dados
2,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,6,38.000,3,Scenario: Cadastrar Entidade com sucesso Given que o usuário está na funcionalidade de cadastro de entidade When ele preenche os dados e confirma o cadastro Then a entidade é cadastrada com sucesso,"Scenario: Cadastrar Entidade com sucesso Given o usuário está na página de cadastro de entidades And o usuário preenche todos os campos obrigatórios com dados válidos When o usuário clica no botão ""Salvar"" Then o sistema deve exibir uma mensagem de confirmação indicando que a entidade foi cadastrada com sucesso And o novo registro deve aparecer na lista de entidades existentes"
3,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,9,38.000,4,Scenario: Cadastrar Entidade com sucesso Given que o usuário está na funcionalidade de cadastro de entidade When ele preenche os dados e confirma o cadastro Then a entidade é cadastrada com sucesso,"Scenario: Cadastrar Entidade com sucesso Given o usuário está na página de cadastro de entidades And o usuário preenche todos os campos obrigatórios com dados válidos When o usuário clica no botão ""Salvar"" Then o sistema deve exibir uma mensagem de confirmação indicando que a entidade foi cadastrada com sucesso And o novo registro deve aparecer na lista de entidades existentes"
4,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,1,39.000,5,Scenario: Cadastrar Entidade com sucesso Given que o usuário está na funcionalidade de cadastro de entidade When ele preenche os dados e confirma o cadastro Then a entidade é cadastrada com sucesso,"Scenario: Cadastrar Entidade com sucesso Given o usuário está na página de cadastro de entidades And o usuário preenche todos os campos obrigatórios com dados válidos When o usuário clica no botão ""Salvar"" Then o sistema deve confirmar a criação da entidade com uma mensagem de sucesso And o novo registro deve aparecer na lista de entidades cadastradas"
5,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,2,39.000,6,Scenario: Cadastrar Entidade com sucesso Given que o usuário está na funcionalidade de cadastro de entidade When ele preenche os dados e confirma o cadastro Then a entidade é cadastrada com sucesso,"Scenario: Cadastro de entidade com sucesso Given o usuário está na página de cadastro de entidade And o usuário preenche todos os campos obrigatórios com dados válidos When o usuário clica no botão ""Salvar"" Then o sistema deve validar os dados e exibir uma mensagem de sucesso And o novo registro da entidade deve ser persistido no banco de dados"
6,ibm-gra

In [ ]:
# ============================================================
# 8. EXPORTAÇÃO DO CSV
# ============================================================

def slug(texto):
    texto = str(texto or "").strip().lower()
    texto = re.sub(r"[^a-z0-9._-]+", "-", texto)
    texto = re.sub(r"-+", "-", texto).strip("-")
    return texto or "sem-identificacao"


metadados_unicos = (
    df_resultados[["modelo", "tecnica"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

if len(metadados_unicos) == 1:
    modelo = metadados_unicos.loc[0, "modelo"]
    tecnica = metadados_unicos.loc[0, "tecnica"]
    nome_csv = f"metricas_manhattan_{slug(modelo)}_{slug(tecnica)}.csv"
else:
    nome_csv = "metricas_manhattan_multiplos_modelos_tecnicas.csv"

# UTF-8 com BOM facilita a abertura correta de acentos no Excel.
df_resultados.to_csv(nome_csv, index=False, encoding="utf-8-sig")

print(f"✓ CSV gerado: {nome_csv}")
print(f"✓ Linhas exportadas: {len(df_resultados):,}")

if BAIXAR_CSV_AUTOMATICAMENTE:
    try:
        from google.colab import files
        files.download(nome_csv)
    except Exception as e:
        print(f"Download automático não realizado: {e}")

✓ CSV gerado: metricas_manhattan_ibm-granite-granite-4.1-8b_few-shot.csv
✓ Linhas exportadas: 2,590


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>